Preparação.

Criação das camadas Bronze, Silver e Gold.

Criação do schema em camada Bronze.

Criação de volume em camada Bronze.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS bronze;
CREATE CATALOG IF NOT EXISTS silver;
CREATE CATALOG IF NOT EXISTS gold;
CREATE SCHEMA IF NOT EXISTS bronze.telecom;
CREATE VOLUME IF NOT EXISTS bronze.telecom.raw;

Realizando download dos dados crus.

In [0]:
import requests, zipfile, os, shutil

os.makedirs('/tmp/anatel', exist_ok=True)
url = 'https://www.anatel.gov.br/dadosabertos/paineis_de_dados/acessos/acessos_banda_larga_fixa.zip'

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open('/tmp/anatel/acessos.zip', 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

with zipfile.ZipFile('/tmp/anatel/acessos.zip') as z:
    csv_files = [f for f in z.namelist() if f.endswith('.csv')]
    print(csv_files)
    z.extractall('/tmp/anatel/')

for csv_file in csv_files:
    shutil.copy(f'/tmp/anatel/{csv_file}',
                f'/Volumes/bronze/telecom/raw/{csv_file}')

Leitura com PySpark, preparação básica para a camada bronze.

In [0]:
df = (spark.read
      .option('header', True)
      .option('sep', ';')
      .option('encoding', 'ISO-8859-1')
      .option('inferSchema', False)
      .csv('/Volumes/bronze/telecom/raw/acessos.csv'))

Inspecionando as tabelas que realizamos o download.

In [0]:
# 1. Listar tudo que está no volume
files = dbutils.fs.ls('/Volumes/bronze/telecom/raw')
for f in sorted(files, key=lambda x: x.name):
    print(f"{f.name:<55} {f.size/1024**2:>8.1f} MB")

In [0]:
# 2. Cabeçalho e primeiras linhas das DUAS variantes do ano mais recente
#    (ajuste o ano para o mais recente que você baixou)
for nome in ['Acessos_Banda_Larga_Fixa_2024.csv',
             'Acessos_Banda_Larga_Fixa_2024_Colunas.csv']:
    print('='*70)
    print(nome)
    print('='*70)
    linhas = (spark.read.text(f'/Volumes/bronze/telecom/raw/{nome}')
              .limit(3).collect())
    for l in linhas:
        print(l.value[:1000])
    print()

Temos um código por municípios que serve como chave.

CNPJ sem pontuação, removendo a necessidade de limpeza.

Provedores regionais de internet são classificados como OUTROS.

Investigação do que há dentro dos arquivos: Acessos_Banda_Larga_Fixa_Total.csv e Densidade_Banda_Larga_Fixa.csv

In [0]:
for nome in ['Acessos_Banda_Larga_Fixa_Total.csv',
             'Densidade_Banda_Larga_Fixa.csv']:
    print('='*70); print(nome); print('='*70)
    for l in spark.read.text(f'/Volumes/bronze/telecom/raw/{nome}').limit(3).collect():
        print(l.value[:600])
    print()

Verificação do último mês disponível dentro da base de dados.

In [0]:
df26 = (spark.read
        .option('header', True).option('sep', ';')
        .csv('/Volumes/bronze/telecom/raw/Acessos_Banda_Larga_Fixa_2026.csv'))

df26.select('Ano', 'Mês').distinct().orderBy('Ano', 'Mês').show(20)

Realizando recorte para o estado do Rio Grande do Sul e último mês dentro do arquivo de dados.

In [0]:
rs = df26.filter((df26['UF'] == 'RS') & (df26['Mês'] == '7'))
print('linhas RS:', rs.count())
print('municípios:', rs.select('Código IBGE Município').distinct().count())
print('CNPJs:', rs.select('CNPJ').distinct().count())
rs.groupBy('Grupo Econômico').count().orderBy('count', ascending=False).show(20)

Análise: total de 497 municípios no estado do RS.

716 CNPJs de empresas distintas que operam dentro do estado do RS.

Arquivo base com 71.608 linhas contidas dentro dos estado do RS.

Criando catálogo de dados

In [0]:
for col in ['Tecnologia', 'Meio de Acesso', 'Tipo de Pessoa',
            'Tipo de Produto', 'Porte da Prestadora', 'Faixa de Velocidade']:
    print(f"\n=== {col} ===")
    rs.groupBy(col).count().orderBy('count', ascending=False).show(30, truncate=False)

Verificação base de CNPJ, mesmo nome de empresa e com CNPJ diferente

In [0]:
from pyspark.sql import functions as F

rs.groupBy('CNPJ').agg(
    F.countDistinct('Empresa').alias('n_nomes'),
    F.countDistinct('Grupo Econômico').alias('n_grupos')
).filter('n_nomes > 1 OR n_grupos > 1').show(20, truncate=False)

Análise: sem nenhuma incosistência de conflito nos CNPJ e nome de empresas repetidos.

Verificação de nulos e zeros na coluna de acessos

In [0]:
rs.select(
    F.count('*').alias('total'),
    F.sum(F.when(F.col('Acessos').isNull(), 1).otherwise(0)).alias('acessos_nulos'),
    F.sum(F.when(F.col('Acessos') == '0', 1).otherwise(0)).alias('acessos_zero'),
    F.sum(F.when(F.col('Código IBGE Município').isNull(), 1).otherwise(0)).alias('ibge_nulo')
).show()

Análise: não possuímos celulas nulas, zeradas ou que precisem ser limpadas.

In [0]:
# Densidade oficial da Anatel para municípios do RS, mês 2026-07
dens = (spark.read.option('header', True).option('sep', ';')
        .csv('/Volumes/bronze/telecom/raw/Densidade_Banda_Larga_Fixa.csv'))

(dens.filter((dens['Ano']=='2026') & (dens['Mês']=='7') & (dens['UF']=='RS'))
     .select('Município', 'Código IBGE', 'Densidade', 'Nível Geográfico Densidade')
     .show(10, truncate=False))

In [0]:
from pyspark.sql import functions as F

print('linhas originais:', rs.count())

grao = rs.groupBy('Código IBGE Município','CNPJ','Tecnologia',
                  'Faixa de Velocidade','Tipo de Pessoa','Tipo de Produto').count()
print('linhas no grão proposto:', grao.count())

# quanto a velocidade exata infla
print('velocidades distintas:', rs.select('Velocidade').distinct().count())